# wf_v4.2 real runs with Llama (diary Nov 10)
Code: GitHub `TriDang2025/attribute-extraction`, branch `wf-v4.2-llama` (from `wf-v4.1`), folder `step1a/`. Results in `results/wf_v42/`.

Model: Llama-3.1-8B-Instruct, bf16, vLLM, temperature 0, guided JSON from each job's schema (setting `llama8b` of `config/wf_v41_models.json`). Runtime: **A100 (40 GB or more)**. Frozen prompts, parsers and scorers.

- Cells 1-5: setup (as the Nov 16 notebook). Cell 6: **T0 smoke test** (1 Step 3 call + 1 Step 4 call); results are committed and pushed.
- Cell 7: **T1** frozen Step 3 A prompt, 18 calls (3 entities x 3 orders x dev50 / dev50b), unchanged (option A of the T0 report). It counts failures; it does not stop on them. Time: about 3 min if all answers stop normally, up to about 45 min if all loop (each looping answer is retried once, unguided, one at a time).
- Cell 8: **T2** Step 4 v4.1 with Llama on input c1 and on input a (Qwen's frozen Step 3 handover), dev50 and dev50b, seeds 0-2: 12 chains x 4 stages = 48 calls. Time: about 5-15 min; up to about 1 h if many answers loop (estimate).
- Cells 9-10: **option B** (user choice 2026-11-11): the same T1 and T2 jobs with "exactly once" guided schemas (`s1a/keyed_v42.py`), frozen prompt text. Cell 9 = Step 3 (18 calls, about 3-5 min), cell 10 = Step 4 (12 chains, about 5-10 min); estimates.
- After a crash or disconnect: rerun cells 1, 3, 4, 5 and then the cell you were on. Answered calls are read from the cache.
- Colab Secrets `HF_TOKEN` and `GH_TOKEN`, notebook access on. The HF account must have accepted the Llama 3.1 license. Never put a token in a cell.
- When the GPU work is done: **Runtime > Disconnect and delete runtime**, so the paid GPU stops.


In [ ]:
#@title 1. Tokens, clone/pull wf-v4.2-llama, git identity
GIT_NAME = "Your Name"  #@param {type:"string"}
GIT_EMAIL = "you@users.noreply.github.com"  #@param {type:"string"}
BRANCH = "wf-v4.2-llama"  #@param {type:"string"}
import os, subprocess
from google.colab import userdata
from huggingface_hub import login

hf_token = (userdata.get("HF_TOKEN") or "").strip()
if hf_token:
    login(token=hf_token)
del hf_token
os.environ["GH_TOKEN"] = (userdata.get("GH_TOKEN") or "").strip()
def run(cmd, **kw): subprocess.run(cmd, check=True, **kw)
run(["git", "config", "--global", "user.name", GIT_NAME])
run(["git", "config", "--global", "user.email", GIT_EMAIL])
run(["git", "config", "--global", "credential.helper",
     '!f() { echo username=token; echo "password=$GH_TOKEN"; }; f'])
REPO = "/content/attribute-extraction"
if not os.path.exists(REPO):
    run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/TriDang2025/attribute-extraction.git", REPO])
else:
    run(["git", "-C", REPO, "checkout", "-q", BRANCH])
    run(["git", "-C", REPO, "pull", "-q", "--rebase", "--autostash"])
%cd /content/attribute-extraction/step1a
!git log -1 --format="code at %h (%an, %ar): %s"

**Cell 2 note:** the install prints dependency-resolver warnings; ignore them. Then **Runtime > Restart session**, rerun cell 1 and skip cell 2.
gliner and sentence-transformers are installed with `--no-deps` so they do not replace the torch / transformers that vLLM needs. Cell 4 checks that they load.

In [ ]:
#@title 2. Install (pinned; a few minutes; then restart the session and skip this cell)
!pip install -q vllm==0.33.0
!pip install -q --no-deps gliner==0.2.29 sentence-transformers==5.6.0
!pip install -q sentencepiece

In [ ]:
#@title 3. Model setting (config/wf_v41_models.json) and GPU check
SETTING = "llama8b"  #@param ["llama8b"]
import sys
sys.path.insert(0, "/content/attribute-extraction/step1a")
from s1a import nov16_v41 as NV
SET = NV.model_setting(SETTING)
print(SETTING, SET)
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "no GPU: change the runtime type to A100"
gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU memory {gb:.1f} GiB")
assert gb >= SET["min_gpu_gb"], f"only {gb:.1f} GiB: {SETTING} needs {SET['min_gpu_gb']} GiB (an A100)"

In [ ]:
#@title 4. Load the model (gpu_memory_utilization leaves room for GLiNER and bge-m3 of loop 2, T6)
MODEL, REVISION = SET["model"], SET["revision"]   # from the setting of cell 3
GPU_UTIL = 0.80  #@param {type:"number"}
MAX_LEN = 32768
import vllm
from vllm import LLM, SamplingParams
print("vllm", vllm.__version__, "| model", MODEL, "| revision", REVISION)
llm = LLM(model=MODEL, revision=REVISION, tokenizer_revision=REVISION, quantization=SET["quantization"],
          max_model_len=MAX_LEN, gpu_memory_utilization=GPU_UTIL)

In [ ]:
#@title 5. Backend: temperature 0, guided JSON per job schema, batches, cache per job (s1a/nov16_v41.VllmBackend)
BATCH = 64  #@param {type:"integer"}
def sampling(max_tokens, schema):
    kw = dict(temperature=0, max_tokens=max_tokens)
    if schema is not None:
        try:
            from vllm.sampling_params import StructuredOutputsParams
            kw["structured_outputs"] = StructuredOutputsParams(json=schema)
        except ImportError:
            from vllm.sampling_params import GuidedDecodingParams
            kw["guided_decoding"] = GuidedDecodingParams(json=schema)
    return SamplingParams(**kw)
backend = NV.VllmBackend(llm, MODEL, REVISION, sampling, max_len=MAX_LEN, batch=BATCH, vllm_version=vllm.__version__)
backend.model_id

In [ ]:
#@title 6. T0 smoke test: 1 Step 3 call (frozen A prompt, hotel dev50b s0) + 1 Step 4 call (v4.1 stage 1, c1 dev50 seed 0)
import subprocess, json
from pathlib import Path
from s1a import batch_v4 as B
OUT = Path("results/wf_v42/t0_smoke")
!python scripts/t0_smoke_v42.py --build
jobs = B.read_json(OUT / "manifest.json")["jobs"]
print(backend.run(OUT, jobs))
r = subprocess.run(["python", "scripts/t0_smoke_v42.py", "--check"], capture_output=True, text=True)
print(r.stdout or r.stderr[-1000:])
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v42 T0 smoke", "wf_v42/t0_smoke"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T0", "PASS" if r.returncode == 0 else "FAIL: stop and tell Claude")


In [ ]:
#@title 7. T1: frozen Step 3 A prompt with Llama, 18 calls (option A: unchanged, count the failures)
import subprocess
from pathlib import Path
from s1a import batch_v4 as B
OUT1 = Path("results/wf_v42/t1_step3")
!python scripts/t1_step3_v42.py --build
print(backend.run(OUT1, B.read_json(OUT1 / "manifest.json")["jobs"]))
!python scripts/t1_step3_v42.py --check
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v42 T1 step3", "wf_v42/t1_step3"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
print("T1 done: tell Claude 'T1 done'")


In [ ]:
#@title 8. T2: Step 4 v4.1 with Llama, inputs c1 and a, dev50 + dev50b, seeds 0-2 (12 chains, 4 stages)
import subprocess
from pathlib import Path
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import t2_step4_v42 as T2
print(T2.run(T2.OUT, backend))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v42 T2 step4", "wf_v42/t2_step4"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 9. Option B, Step 3: the 18 T1 calls with keyed "exactly once" schemas (s1a/keyed_v42.py)
import subprocess
from pathlib import Path
from s1a import batch_v4 as B
from s1a import keyed_v42 as K
OUT1B = Path("results/wf_v42/t1b_step3_keyed")
!python scripts/t1_step3_v42.py --build --keyed
print(K.KeyedLLM(backend).run(OUT1B, B.read_json(OUT1B / "manifest.json")["jobs"]))
!python scripts/t1_step3_v42.py --check --keyed
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v42 T1b keyed", "wf_v42/t1b_step3_keyed"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])


In [ ]:
#@title 10. Option B, Step 4: the 12 T2 chains with keyed "exactly once" schemas for 4.1, 4.2, 4.4
import sys, subprocess
sys.path.insert(0, "/content/attribute-extraction/step1a/scripts")
import t2_step4_v42 as T2
print(T2.run(T2.OUT_KEYED, backend, keyed=True))
p = subprocess.run(["bash", "colab/sync.sh", "push-results", "wf_v42 T2b keyed", "wf_v42/t2b_step4_keyed"], capture_output=True, text=True)
print(p.stdout.strip() or p.stderr[-500:])
